Install required libraries and restart kernel:

In [1]:
%pip install tensorflow==2.16.2
%pip install matplotlib==3.9.2
%pip install numpy==1.26.4
%pip install scipy==1.14.1
%pip install scikit-learn==1.5.2
%pip install tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Import libraries:

In [2]:
# Standard library imports
import os
import zipfile
import urllib.request
import logging
from collections import Counter
from datetime import datetime
from typing import Optional
from pathlib import Path

# Third-party imports
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# TensorFlow imports
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from tensorflow.keras.applications import VGG16, MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, 
    Flatten, 
    Dropout, 
    BatchNormalization, 
    GlobalAveragePooling2D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.mixed_precision import set_global_policy

In [3]:
# Logger setup
_LOGGER = logging.getLogger(__name__)


def setup_logging(debug: bool = True) -> None:
    """Configure logging with proper timestamp formatting."""
    level = logging.DEBUG if debug else logging.INFO
    logging.basicConfig(
        level=level,
        format="%(asctime)s %(levelname)s: %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    _LOGGER.setLevel(level)


# Initialize logging 
setup_logging(debug = True)

Download and extract dataset

In [4]:
def download_dataset(url: str, output_file: str) -> Path:
    """
    Download a dataset from a URL with tqdm progress bar and logging.

    Creates parent directories if needed, shows real-time download progress
    via tqdm (bytes, unit scaling, rate), and logs start/completion with final size.

    Args:
        url (str): Remote URL to download (e.g., ZIP dataset).
        output_file (str): Local path to save file (str or Path-like).

    Returns:
        Path: Path to the downloaded file.

    Raises:
        urllib.error.URLError: If download fails (network, invalid URL).
        OSError: If file write fails (permissions, disk space).

    Example:
        >>> download_dataset("https://example.com/data.zip", "data/fruits.zip")
        PosixPath('data/fruits.zip')
    """
    output_file = Path(output_file)
    _LOGGER.info(f"📥 Downloading {url}...")
    
    output_file.parent.mkdir(parents=True, exist_ok=True)
    
    def tqdm_hook(t):
        last_b = [0]
        def inner(b=1, bs=1, tsize=None):
            downloaded = b * bs
            t.update(downloaded - last_b[0])
            last_b[0] = downloaded
        return inner
    
    urllib.request.urlretrieve(url, str(output_file), reporthook=tqdm_hook(tqdm(unit='B', unit_scale=True)))
    
    file_size = os.path.getsize(output_file) / (1024**2)
    _LOGGER.info(f"✅ Download COMPLETE ({file_size:.1f} MB)")
    return output_file



def extract_zip_in_chunks(
    zip_path: str, 
    extract_to: str, 
    batch_size: int = 1000,
    overwrite: bool = False
) -> None:
    """
    Extract large ZIP file in memory-efficient batches with progress logging.

    Processes files in batches to avoid loading the entire ZIP index into memory.
    Skips directories and existing files (unless overwrite=True). Logs batch progress.

    Args:
        zip_path (str): Path to input ZIP file (must exist).
        extract_to (str): Root directory to extract contents (created if needed).
        batch_size (int): Number of files to extract per batch (default: 1000).
        overwrite (bool): Extract even if files exist (default: False).

    Returns:
        None

    Raises:
        FileNotFoundError: If zip_path does not exist.
        zipfile.BadZipFile: If ZIP is corrupted/invalid.
        PermissionError: If cannot write to extract_to.

    Example:
        >>> extract_zip_in_chunks("data/fruits.zip", "data/extracted", batch_size=500)
        Extracts fruits-360 dataset in 500-file chunks.
    """
    zip_path = Path(zip_path)
    extract_to = Path(extract_to)
    
    if not zip_path.exists():
        raise FileNotFoundError(f"❌ ZIP not found: {zip_path}")
    
    _LOGGER.info(f"📦 Extracting {zip_path} → {extract_to} (batch={batch_size})")
    extract_to.mkdir(parents=True, exist_ok=True)
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        file_list = zip_ref.namelist()
        total_files = len(file_list)
        _LOGGER.debug(f"Found {total_files} files")
        
        for i in range(0, total_files, batch_size):
            batch = file_list[i:i + batch_size]
            extracted_count = 0
            
            for file_path in batch:
                if file_path.endswith('/'):
                    continue
                
                target_path = extract_to / file_path
                if overwrite or not target_path.exists():
                    target_path.parent.mkdir(parents=True, exist_ok=True)
                    zip_ref.extract(file_path, extract_to)
                    extracted_count += 1
            
            progress = (i + batch_size) / total_files * 100
            _LOGGER.info(
                f"📊 Batch {i//batch_size + 1}: "
                f"{min(i + batch_size, total_files)}/{total_files} "
                f"({progress:.1f}%)"
            )
    
    _LOGGER.info(f"✅ Extraction done: {extract_to}")


def prepare_dataset(
    dataset_url: str,
    zip_path: str,
    extract_path: str,
    fallback_urls: list[str] | None = None,
    force_download: bool = False,
    force_extract: bool = False,
    batch_size: int = 1000,
    auto_cleanup: bool = False,
) -> tuple[Path, Path]:
    """
    Download and extract dataset with fallback URLs, retries, and cleanup.

    Tries primary URL + fallbacks until success. Downloads if ZIP missing/forced,
    extracts in chunks if dir missing/forced. Cleans up ZIP post-extraction if enabled.
    Handles failures gracefully, raises only if all URLs fail.

    Args:
        dataset_url (str): Primary URL for ZIP download.
        zip_path (str): Local path to save ZIP (created if missing).
        extract_path (str): Dir to extract contents (created if missing).
        fallback_urls (list[str] | None): Backup URLs if primary fails (default: None).
        force_download (bool): Re-download even if ZIP exists (default: False).
        force_extract (bool): Re-extract even if dir exists (default: False).
        batch_size (int): Files per extract batch (default: 1000).
        auto_cleanup (bool): Delete ZIP after successful extraction (default: False).

    Returns:
        tuple[Path, Path]: (zip_path, extract_path) - paths to ZIP and extracted data.

    Raises:
        RuntimeError: If all URLs fail to download.
        urllib.error.URLError: Download failures (propagated from download_dataset).
        zipfile.BadZipFile: Corrupted ZIP (from extract_zip_in_chunks).

    Example:
        >>> zip_p, ext_p = prepare_dataset(
        ...     "https://example.com/fruits.zip",
        ...     "data/fruits.zip",
        ...     "data/fruits-extracted",
        ...     fallback_urls=["https://backup.com/fruits.zip"],
        ...     auto_cleanup=True
        ... )
        >>> ext_p / "Training"  # Contains extracted images
        PosixPath('data/fruits-extracted/Training')
    """

    zip_path = Path(zip_path)
    extract_path = Path(extract_path)
    
    urls_to_try = [dataset_url]
    if fallback_urls:
        urls_to_try.extend([u for u in fallback_urls if u != dataset_url])
    
    for url in urls_to_try:
        try:
            _LOGGER.info(f"🔄 Attempting: {url}")
            
            if force_download or not zip_path.exists():
                zip_path = download_dataset(url, zip_path)
            
            if force_extract or not extract_path.exists():
                extract_zip_in_chunks(zip_path, extract_path, batch_size)
            
            _LOGGER.info(f"✅ SUCCESS: {url}")
            break
            
        except Exception as e:
            _LOGGER.warning(f"❌ Failed {url}: {str(e)[:100]}...")
            continue
    else:
        raise RuntimeError(f"All {len(urls_to_try)} URLs failed!")
    
    if auto_cleanup and zip_path.exists() and extract_path.exists():
        zip_path.unlink()
        _LOGGER.info("🧹 ZIP cleaned up")
    
    _LOGGER.info(f"🎉 Dataset ready → {zip_path} | {extract_path}")
    return zip_path, extract_path


In [5]:
# Dataset configuration
IBM_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/4yIRGlIpNfKEGJYMhZV52g/fruits-360-original-size.zip"
GITHUB_URL = "https://github.com/fruits-360/fruits-360-original-size/archive/refs/heads/main.zip"
LOCAL_ZIP = "fruits-360-original-size.zip"
EXTRACT_DIR = "fruits-360-original-size"

# Download and extract dataset
zip_file, data_dir = prepare_dataset(
    dataset_url=GITHUB_URL,
    zip_path=LOCAL_ZIP,
    extract_path=EXTRACT_DIR,
    fallback_urls=[IBM_URL]
)


2026-03-08 12:02:09 INFO: 🔄 Attempting: https://github.com/fruits-360/fruits-360-original-size/archive/refs/heads/main.zip
2026-03-08 12:02:09 INFO: 📥 Downloading https://github.com/fruits-360/fruits-360-original-size/archive/refs/heads/main.zip...
3.96GB [28:11, 2.34MB/s]
2026-03-08 12:30:20 INFO: ✅ Download COMPLETE (3779.7 MB)
2026-03-08 12:30:20 INFO: 📦 Extracting fruits-360-original-size.zip → fruits-360-original-size (batch=1000)
2026-03-08 12:30:20 DEBUG: Found 96609 files
2026-03-08 12:30:21 INFO: 📊 Batch 1: 1000/96609 (1.0%)
2026-03-08 12:30:21 INFO: 📊 Batch 2: 2000/96609 (2.1%)
2026-03-08 12:30:22 INFO: 📊 Batch 3: 3000/96609 (3.1%)
2026-03-08 12:30:23 INFO: 📊 Batch 4: 4000/96609 (4.1%)
2026-03-08 12:30:24 INFO: 📊 Batch 5: 5000/96609 (5.2%)
2026-03-08 12:30:24 INFO: 📊 Batch 6: 6000/96609 (6.2%)
2026-03-08 12:30:25 INFO: 📊 Batch 7: 7000/96609 (7.2%)
2026-03-08 12:30:25 INFO: 📊 Batch 8: 8000/96609 (8.3%)
2026-03-08 12:30:26 INFO: 📊 Batch 9: 9000/96609 (9.3%)
2026-03-08 12:30:27 

In [6]:
# Directory structure 

# Ensure that your data set is organized as follows: 


# dataset/
# ├── train/
# │   ├── Class1/
# │   ├── Class2/
# │   ├── Class3/
# │   └── (other classes...)
# ├── val/
# │   ├── Class1/
# │   ├── Class2/
# │   ├── Class3/
# │   └── (other classes...)
# └── test/
#     ├── Class1/
#     ├── Class2/
#     ├── Class3/
#     └── (other classes...)


# Each subdirectory under train and val should contain images of the respective fruit category. 